# Solution D2 · Four ways to measure importance: impurity, permutation, drop-column and group

**Type**: Interpret

## Objective

On a designed dataset (two independent informative features, **a group of three correlated features** carrying the same information, a continuous noise column and a binary one) compare four importance measures of a random forest: impurity, permutation on the test set, **drop-column** (retrain without the feature and see
how much the accuracy drops) and group permutation. Decide which method answers which question.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split


def generate(n=3000, seed=0):
    rng = np.random.default_rng(seed)
    x0, x1, z = rng.normal(size=(3, n))
    g = np.column_stack([z + rng.normal(0, 0.15, n) for _ in range(3)])           # three noisy versions of the same factor
    y = (1.2 * x0 + 0.8 * x1 + 1.5 * z + rng.normal(0, 0.8, n) > 0).astype(int)
    X = np.column_stack([x0, x1, g, rng.normal(size=n), rng.integers(0, 2, n)])
    return X, y


names = ["x0", "x1", "g0", "g1", "g2", "noise_c", "noise_b"]
X, y = generate()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
rf = lambda: RandomForestClassifier(200, random_state=0, n_jobs=-1)
f = rf().fit(X_tr, y_tr)
full_acc = f.score(X_te, y_te)
print("accuracy of the full model:", round(full_acc, 3))

## Commented solution

In [ ]:
result = permutation_importance(f, X_te, y_te, n_repeats=10, random_state=0)
table = pd.DataFrame(index=names, columns=["impurity", "permutation", "drop_column"], dtype=float)
table["impurity"] = f.feature_importances_
table["permutation"] = result.importances_mean
for j, name in enumerate(names):
    without = rf().fit(np.delete(X_tr, j, axis=1), y_tr).score(np.delete(X_te, j, axis=1), y_te)          # retrain without the column
    table.loc[name, "drop_column"] = full_acc - without
table.round(3)

In [ ]:
group = [2, 3, 4]
rng = np.random.default_rng(0)
losses = []
for _ in range(10):
    p = rng.permutation(len(X_te))
    Xp = X_te.copy()
    Xp[:, group] = X_te[p][:, group]                                          # the three columns together, same permutation
    losses.append(full_acc - f.score(Xp, y_te))
group_permutation = float(np.mean(losses))
group_drop = float(full_acc - rf().fit(np.delete(X_tr, group, axis=1), y_tr).score(np.delete(X_te, group, axis=1), y_te))
print(f"group g0-g1-g2: permutation {group_permutation:.3f} | drop-column {group_drop:.3f}")

In [ ]:
impurity_rewards_noise = bool(table.loc["noise_c", "impurity"] > 0.05)
g = ["g0", "g1", "g2"]
single_permutation_underestimates_group = bool(table.loc[g, "permutation"].sum() < 0.5 * group_permutation)
drop_column_underestimates_group = bool(table.loc[g, "drop_column"].sum() < 0.5 * group_drop)
drop_candidates = list(table.index[table["drop_column"] <= 0.005])
selection_error = [n for n in g if n in drop_candidates]
print("removable according to the drop-column, one at a time:", drop_candidates)
print("... of which belonging to the informative group:", selection_error)

## Check

In [ ]:
with verify():
    check(list(table.index) == names and table.notna().all().all() and close(table["impurity"].sum(), 1.0), "complete table; the impurity importances sum to 1", "seven rows and three columns")
    check(table.loc["noise_c", "impurity"] > 0.05 and table.loc["noise_c", "permutation"] < 0.01 and impurity_rewards_noise is True, "the continuous noise is worth over 5% by impurity and less than 0.01 by permutation", "impurity rewards features with many distinct values")
    check(table.loc[["g0", "g1", "g2"], "drop_column"].max() < 0.01 and table.loc[["g0", "g1", "g2"], "impurity"].min() > 0.1, "each copy has a drop-column below 0.01 (the others replace it) but an impurity importance over 0.1", "the copies are interchangeable")
    check(group_permutation > 0.06 and group_drop > 0.05 and single_permutation_underestimates_group is True and drop_column_underestimates_group is True, "the group together is worth over 0.06 (permutation) and 0.05 (drop); the sum of the single importances is worth less than half of it", "permute or remove the three columns together")
    check("noise_c" in drop_candidates and "noise_b" in drop_candidates and set(selection_error) == {"g0", "g1", "g2"}, "the drop-column alone flags as removable also all three copies of the informative group: selection error", "drop-column <= 0.005")
done()

## What you learned

- **The impurity rewards the continuous noise** (`noise_c` is worth 5.8% of importance: many thresholds to try) while by permutation and drop-column it is zero: they are methods that measure how much the **model gets worse** on new data by removing or shuffling that column.
- **With three copies of the same factor, each one looks useless** with the drop-column (−0.008, −0.013, 0.001: removing one leaves the other two) and is worth little with the single permutation (0.06, 0.02, 0.01, in total less than half of the 0.25 of the group), yet the group is worth a lot. The three columns together carry the factor `z`, a strong predictor of the problem: removing them all would cost about 18 points of accuracy (0.878 → 0.70).
- **Typical selection error:** a rule "remove the features with drop-column ~0" applied to a group of correlated features removes them all (one at a time each is redundant) and the information is lost. The right rule is to remove **one at a time, retraining**, or to reason by groups.
- Summary of the questions: **impurity** = "what does the tree use on the training set?" (fast, biased); **permutation** = "what does the model lean on to predict?" (underestimates groups); **drop-column** = "can I remove this column?" (expensive: one retraining per column; underestimates groups); **group** = "how much is this information worth?".

## Common pitfall

Calling a number "importance" without saying which question it answers. The colleague who uses `feature_importances_` to "see what is needed" and the one who uses drop-column to "see what to remove" can reach opposite conclusions, and both are right for their own question.

## Interview question

<details><summary>How do you choose which features to remove from a model?</summary>

Not from the importance alone: impurity is biased toward features with many values, permutation and drop-column underestimate groups of correlated features. I group the redundant variables, remove one group (or one feature) at a time retraining and checking the performance on validation data,
and keep the smallest set that does not get worse. The final cut also depends on the cost of collecting and maintaining each feature.

</details>